### How to read a url file in pyspark, read zip file from url in pyspark databricks

In [0]:
!pip install urllib

ERROR: Could not find a version that satisfies the requirement urllib
ERROR: No matching distribution found for urllib
You should consider upgrading via the '/local_disk0/.ephemeral_nfs/envs/pythonEnv-a27240b6-eabb-4eed-bfe8-9349330b55fc/bin/python -m pip install --upgrade pip' command.


### Request remote zip file using urllib.request.Request

In [0]:
import urllib

urllib.request.urlretrieve("https://resources.lendingclub.com/LoanStats3a.csv.zip", "/tmp/LoanStats3a.zip")


Out[2]: ('/tmp/LoanStats3a.zip', <http.client.HTTPMessage at 0x7fe0117a5700>)

##### Same way done for gz file also

### Use sh command to unzip the file

- Unzip file into default drive
file:/databricks/driver/


- One way we can do unzip in sh command, another way we can move this to dbfs location and do the same

- To unzip gzip

%sh
ginzip -r <zipfilepath with file>

In [0]:
%sh
unzip /tmp/LoanStats3a.zip
tail -n +2 LoanStats3a.csv > LoanStats3a_unzip.csv
rm LoanStats3a.csv

Archive:  /tmp/LoanStats3a.zip
  inflating: LoanStats3a.csv         


In [0]:
%fs

ls  file:/databricks/driver/

path,name,size
file:/databricks/driver/conf/,conf/,4096
file:/databricks/driver/preload_class.lst,preload_class.lst,813069
file:/databricks/driver/eventlogs/,eventlogs/,4096
file:/databricks/driver/metastore_db/,metastore_db/,4096
file:/databricks/driver/temp.csv,temp.csv,42408111
file:/databricks/driver/LoanStats3a_unzip.csv,LoanStats3a_unzip.csv,42408111
file:/databricks/driver/logs/,logs/,4096
file:/databricks/driver/ganglia/,ganglia/,4096


### Use dbfs utilities to move unzippped file from default location to /tmp locatio

In [0]:
dbutils.fs.mv("file:/databricks/driver/LoanStats3a_unzip.csv","dbfs/tmp/LoanStats3a_unzip.csv" )

Out[8]: True

In [0]:
%fs

ls dbfs/tmp/

path,name,size
dbfs:/dbfs/tmp/LoanStats3a_unzip.csv,LoanStats3a_unzip.csv,42408111


In [0]:
%fs

ls  file:/databricks/driver/

path,name,size
file:/databricks/driver/conf/,conf/,4096
file:/databricks/driver/preload_class.lst,preload_class.lst,813069
file:/databricks/driver/eventlogs/,eventlogs/,4096
file:/databricks/driver/metastore_db/,metastore_db/,4096
file:/databricks/driver/temp.csv,temp.csv,42408111
file:/databricks/driver/logs/,logs/,4096
file:/databricks/driver/ganglia/,ganglia/,4096


- No file in default now, its moved

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/dbfs/tmp/LoanStats3a_unzip.csv")

In [0]:
display(df)

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,desc,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,next_pymnt_d,last_credit_pull_d,collections_12_mths_ex_med,mths_since_last_major_derog,policy_code,application_type,annual_inc_joint,dti_joint,verification_status_joint,acc_now_delinq,tot_coll_amt,tot_cur_bal,open_acc_6m,open_act_il,open_il_12m,open_il_24m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,sec_app_mths_since_last_major_derog,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
null,null,5000,5000,4975.0,36 months,10.65%,162.87,B,B2,null,10+ years,RENT,24000.0,Verified,Dec-2011,Fully Paid,n,null,Borrower added on 12/22/11 > I need to upgrade my business technologies.,credit_card,Computer,860xx,AZ,27.65,0,Jan-1985,1,null,null,3,0,13648,83.7%,9,f,0.00,0.00,5863.1551866952,5833.84,5000.00,863.16,0.0,0.0,0.0,Jan-2015,171.62,null,Jul-2021,0,null,1,Individual,null,null,null,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null,null,null,null,null,null,null,null,null,N,null,null,null,null,null,null
null,null,2500,2500,2500.0,60 months,15.27%,59.83,C,C4,Ryder,< 1 year,RENT,30000.0,Source Verified,Dec-2011,Charged Off,n,null,Borrower added on 12/22/11 > I plan to use this money to finance the motorcycle i am looking at. I plan to have it paid off as soon as possible/when i sell my old bike. I only need this money because the deal im looking at is to good to pass up. Borrower added on 12/22/11 > I plan to use this money to finance the motorcycle i am looking at. I plan to have it paid off as soon as possible/when i sell my old bike.I only need this money because the deal im looking at is to good to pass up. I have finished college with an associates degree in business and its takingmeplaces,car,bike,309xx,GA,1,0,Apr-1999,5,null,null,3,0,1687,9.4%,4,f,0.00,0.00,1014.53,1014.53,456.46,435.17,0.0,122.9,1.11,Apr-2013,119.66,null,Oct-2016,0,null,1,Individual,null,null,null,0,null,null,null

- Here got all loan related data